# **CREACIÓN DEL ENTORNO (.venv)**   
Para que funcione correctamente:
1. Instalamos python la versión 3.13.12 (es la que he probado yo)

2. Creamos un entorno virtual (recomendado para no tener trescientos paquetes instalados)
    - Haciendo Crtl+Shift+P y seleccionando "Python: Create Environment".
    - Te tiene que haber creado una dirección en la carpeta .venv.
    - Podemos comprobarlo abriendo una terminal y ponemos "pip --version" y tiene que salir la dirección de la carpeta .venv.

# **BIBLIOTECAS NECESARIAS**

In [11]:
# Importamos las librerias necesarias
from googleapiclient.discovery import build
from yt_dlp import YoutubeDL
import pprint as pprint
import re
import glob
import os
import json

import numpy as np
import pandas as pd
from pandas import DataFrame

c:\Users\Marina\Documents\Uni\ProyectoDatos\c2526-R1\.venv\Lib\site-packages\requests\__init__.py:113: RequestsDependencyWarning: urllib3 (2.6.3) or chardet (7.0.1)/charset_normalizer (3.4.4) doesn't match a supported version!
  warnings.warn(


# **GUARDAR API KEY Y CREAR EL "SERVIDOR"**
La función build nos construye un objeto de la API de Youtube, que usaremos para que llame a la API y acceder a los datos.

In [12]:
# Guardamos nuestra API_KEY leyendo de la carpeta privada
with open("../Private/Claves.json", "r", encoding="utf-8") as archivo:
    claves = json.load(archivo)
API_KEY = claves["Clave_API"]
if API_KEY:
    print("Loaded API_KEY succesfully")

Loaded API_KEY succesfully


In [5]:
# Construimos el objeto de la API de YouTube
youtube = build('youtube', 'v3', developerKey=API_KEY)

# **FUNCIÓN PARA LIMPIAR EL TEXTO**

In [6]:
#Función temporal para limpiar texto
import re

def clean_vtt(text):
    text = re.sub(r"WEBVTT.*\n", "", text)
    text = re.sub(r"\d+:\d+:\d+\.\d+ --> .*", "", text)
    text = re.sub(r"<.*?>", "", text)
    text = re.sub(r"\n+", "\n", text)
    return text.strip()

def clean_vtt_smart(text):
    lines = []
    for line in text.splitlines():
        line = line.strip()
        if not line:
            continue
        if not lines or not line in lines[-1]:
            lines.append(line)
    return " ".join(lines)

# **GENERAMOS UNA PEQUEÑA BASE DE DATOS**
Lo guardaremos en un archivo csv, guardando la información de 50 videos.


In [7]:
# Lista de los id de los videos (usando el id el coste de la consulta es el mmínimo)
list_id = [
    'y5qZyNjjd4A', '-EOCVYvDY14', 'xtvM1BeGhXQ', 'TK0ERlbF_jE', 'wQK47p3lsn8', 'v9M7g-nKMBU', '5FRA7UXlGu8', 'TEA-hKkWDKM', 'Z-tbr5L0XDE', 'fLJBzhcSWTk', 
    'mDbpbPHW8bg', 'um1-gqoCJNA', 'Lmu6z4aj3Vk', 'WXhZGtVvwUo', 'Yasyj-0pjRI', '9shK1SmrWsU', '6qhuPwTcc50', 'nBdMdTpyPqk', 'UiCJhSuLdok', 'APD3ESDsqL0', 
    'DAv10RDdr9U', 'Cdg9UbVD9yo', 'FT-C0zSKQH8', 'C9VC6kt1ors', 'vZnPo0fPqqY', 'UplwT_a1IT8', 'itl1Lf4hq7c', 'cAeDOt8kI4g', 'JF5x6qgdi-8', 'E4p3-IpaIzE', 
    '6izsSWPPYsc', 'eTJQFu9JK6A', '80c6rdulV8o', 'Jp24nGUPoR0', 'pq330xEyTuY', 'vfIupidHgQg', 'dAVJO18j-do', 'uKzjZu_KVzU', 'WgIpM-4ZX88', 'PrBUjXaRSUQ', 
    '-G0u13COALA', 'sxDHyYpQNBk', 'dRBowc83jVs', '4k_2sjvnasE', '3Wn69w0W9C8', '8KIYBNbbK4E', 'P4QAFljGP7s', '_G6NtIVZhRE', 'cPN4H0sSCHQ', 'YTbRsGRWZW0'
    ]

# Creo el dataframe vacío
df_data = pd.DataFrame({
    "ID": [],
    "Titulo": [],
    "Descripcion": [],
    "Visualizaciones": [],
    "Numero_likes": [],
    "Duracion": [],
    "Fecha_publicacion": [],
    "Titulo_canal": [],
    "Subtitulos": []
})
df_data

,ID,Titulo,Descripcion,Visualizaciones,Numero_likes,Duracion,Fecha_publicacion,Titulo_canal,Subtitulos


In [8]:
from urllib import response


def get_info(id_video):
    """ Saca la información del video, si hay subtítulos los limpia, y añade dichos datos al dataframe. """
    
    # Inicializamos la variable de los subtitulos
    cleant_sub = None

    # Hacemos la llamada a la API para obtener los detalles del video
    request = youtube.videos().list(
        part="snippet,contentDetails,statistics,status,topicDetails,recordingDetails",
        id=id_video
    )  

    # Ejecutamos la solicitud
    response = request.execute()

    if not response["items"]:
        return None
    
    video = response["items"][0]

    has_captions = video["contentDetails"].get("caption") in ["true", True]

    # --- SUBTÍTULOS ---
    if (video["contentDetails"].get("caption") == "true"):
        url =  "https://www.youtube.com/watch?v=" + id_video
        # Opciones de descarga
        ydl_opts = {
            "skip_download": True,
            "writesubtitles": True,
            "writeautomaticsub": True,      # subtítulos automáticos
            "subtitleslangs": ["en"], # idioma
            "subtitlesformat": "vtt",       # formato
            "outtmpl": f"subs/{id_video}.%(ext)s",
            "quiet": True
        }

        with YoutubeDL(ydl_opts) as ydl:
            ydl.download([url])

        vtt_file = f"subs/{id_video}.en.vtt"

        if os.path.exists(vtt_file):
            with open(vtt_file, "r", encoding="utf-8") as f:
                subtitles = f.read()

            clean_sub = clean_vtt(subtitles)
            cleant_sub = clean_vtt_smart(clean_sub)

            # Borramos el archivo de subtítulos descargado tras limpiarlo
            # os.remove(vtt_file)
    
    # --- GENEROS ---
    generos = video.get("topicDetails", {}).get("topicCategories", [])

    if generos:
        generos = [
            genre.split("/")[-1].replace("_", " ")
            for genre in generos
        ]
        generos_str = ", ".join(generos)
    
    else:
        generos_str = "None"

    # --- DATAFRAME ---
    df_video = pd.DataFrame({
        "ID": id_video,
        "Titulo": video["snippet"]["title"],
        "Descripcion": video["snippet"]["description"],
        "Visualizaciones": video["statistics"]["viewCount"],
        "Numero_likes": video["statistics"].get("likeCount", None),
        "Duracion": video["contentDetails"]["duration"],
        "Fecha_publicacion": video["snippet"]["publishedAt"],
        "Titulo_canal": video["snippet"]["channelTitle"],
        "Subtitulos": cleant_sub if has_captions and cleant_sub else "None",
        "Generos": generos_str
    }, index=[0])

    return df_video

In [9]:
df_videos = []

for id in list_id:
    df_videos.append(get_info(id))

df_data = pd.concat(df_videos, ignore_index=True)

In [10]:
df_data

,ID,Titulo,Descripcion,Visualizaciones,Numero_likes,Duracion,Fecha_publicacion,Titulo_canal,Subtitulos,Generos
0,y5qZyNjjd4A,Orishas- El kilo (High Quality) Official Video,DISCLAIMER: music and video belongs to its res...,1816640,16525,PT3M45S,2011-02-15T11:01:25Z,philippekogler,None,"Hip hop music, Music, Music of Latin America, ..."
1,-EOCVYvDY14,How Jynxzi Saved Rocket League *JYNXZI REACTS*,How Jynxzi Saved Rocket League\n\nOriginal Cre...,757388,16441,PT29M11S,2026-02-03T00:19:05Z,Jynxzi Live,None,"Racing video game, Sports game, Video game cul..."
2,xtvM1BeGhXQ,Evento “Together for Democracy” de @democracyf...,,10459,1122,PT2M54S,2026-02-02T19:28:41Z,JmMonteBlack,None,"Politics, Society"
3,TK0ERlbF_jE,Guzo - Guzo - Kingdom Sound Live Production Al...,Guzo - Guzo - Kingdom Sound Live Production Al...,1439172,20394,PT13M4S,2025-08-16T04:30:16Z,Kingdom Sound,"Kind: captions Language: en Though He was God,...","Christian music, Music, Religion"
4,wQK47p3lsn8,How to become King of an existing Kingdom in M...,If founding your own kingdom sounds like too m...,1148180,26470,PT41S,2025-05-18T17:00:35Z,TaleWorlds Entertainment,None,"Action-adventure game, Action game, Role-playi..."
5,v9M7g-nKMBU,Holy Diver,Provided to YouTube by Rhino/Warner Records\n\...,24061620,288074,PT5M54S,2014-11-08T22:46:06Z,Dio - Topic,None,"Music, Rock music"
6,5FRA7UXlGu8,Heaven and Hell (2021 Remaster),Provided to YouTube by Rhino/Warner Records\n\...,3473767,40274,PT6M58S,2021-03-05T03:32:47Z,Black Sabbath - Topic,None,"Music, Rock music, Soul music"
7,TEA-hKkWDKM,"Decades of Anti-Immigration Policy Created ""Ma...",Support our work: https://democracynow.org/don...,137132,3191,PT14M6S,2026-01-28T15:54:32Z,Democracy Now!,None,"Politics, Society"
8,Z-tbr5L0XDE,Kilo,Provided to YouTube by Ditto Music\n\nKilo · Y...,844820,11593,PT4M8S,2018-06-07T21:42:02Z,Yung Bleu - Topic,None,"Hip hop music, Music"
9,fLJBzhcSWTk,Why Socrates Hated Democracy,We’re used to thinking hugely well of democrac...,14092867,589519,PT4M22S,2016-11-28T14:00:02Z,The School of Life,Kind: captions Language: en We are used to thi...,Knowledge


Una vez que tenemos el dataframe, lo exportamos en formato csv.

In [ ]:
data_csv = df_data.to_csv("data_videos.csv", index=False)

In [8]:
get_info("T_-XrleDgTI")

,ID,Titulo,Descripcion,Visualizaciones,Numero_likes,Duracion,Fecha_publicacion,Titulo_canal,Subtitulos,Generos
0,T_-XrleDgTI,"""Out for Love"" - Hazbin Hotel (Sing along) Sub...",#hazbinhotel #amazonprimevideo #vivziepop #hel...,3824,71,PT1M27S,2024-02-02T02:04:10Z,Alejo Muñoz,None,"Music, Pop music"


In [ ]:
import Server_PD

In [2]:
import os
print(os.getcwd())

c:\Users\Marina\Documents\Uni\ProyectoDatos\c2526-R1\src\notebooks
